# UTS Graf Sains Data

| Nama | NIM | Kelas |
|---|---|---|
| Muh. Hilmy Fahrizal | 23031554125 | 2023D |

## 1. Pendahuluan

### 1.1 Persiapan dan Perkenalan Lingkungan

Pengerjaan memakai *virtual environment* agar paket proyek ini terpisah dari paket lain di komputer:

```bash
python -m venv myenv              # 1. membuat environment baru
conda deactivate                  # 2. mematikan environment lain (jika ada)
source myenv/Scripts/activate     #    mengaktifkan myenv (Linux/macOS: myenv/bin/activate)
pip install -r requirements.txt   # 3. memasang dependensi dari requirements.txt
```

### 1.2 Mempelajari Penggunaan Driver Neo4j

Pada bagian ini mempelajari driver Python untuk Neo4j: membuat simpul dan relasi, membaca datanya, lalu menjawab pertanyaan *checkpoint*. Alur pengerjaannya:

1. **Koneksi:** menentukan alamat server (`URI`) dan kredensial (`AUTH`).
2. **Menulis data:** fungsi `add_friend` membuat simpul `Person` dan relasi `KNOWS`.
3. **Membaca data:** fungsi `print_friends` menampilkan teman dari seorang `Person`.
4. **Menjawab checkpoint:** tiga pertanyaan dijawab beserta buktinya dari Neo4j.

#### Langkah 1: Koneksi ke Neo4j

- `URI = "neo4j://localhost:7687"`: protokol `neo4j://` ke server lokal pada port 7687.
- `AUTH`: username dan password. Password diminta saat sel dijalankan.

In [1]:
import getpass
import os

from neo4j import GraphDatabase, RoutingControl

URI = "neo4j://localhost:7687"
AUTH = ("neo4j", os.getenv("NEO4J_PASSWORD") or getpass.getpass("Password Neo4j: "))

#### Langkah 2: Membuat simpul dan relasi (`add_friend`)

Fungsi `add_friend(driver, name, friend_name)` menyimpan satu pertemanan ke dalam graf. Query Cypher di dalamnya terdiri atas tiga perintah `MERGE`:

| Perintah | Arti |
|---|---|
| `MERGE (a:Person {name: $name})` | mencari simpul `Person` dengan nama tersebut, dan **membuatnya jika belum ada** |
| `MERGE (friend:Person {name: $friend_name})` | hal yang sama untuk teman |
| `MERGE (a)-[:KNOWS]->(friend)` | menghubungkan keduanya dengan relasi `KNOWS` yang **berarah** dari `a` ke `friend`, dibuat jika belum ada |

Hal yang perlu diperhatikan:
- `$name` dan `$friend_name` adalah **parameter query**. Nilainya dikirim terpisah lewat `execute_query(..., name=..., friend_name=...)`, sehingga query aman dan dapat dipakai ulang.
- `database_="neo4j"` menentukan database tujuan.
- Karena memakai `MERGE` (bukan `CREATE`), fungsi ini bersifat **idempoten**: dijalankan berulang kali tidak akan menggandakan simpul maupun relasi.

In [2]:
def add_friend(driver, name, friend_name):
    driver.execute_query(
    "MERGE (a:Person {name: $name}) "
    "MERGE (friend:Person {name: $friend_name}) "
    "MERGE (a)-[:KNOWS]->(friend)",
    name=name, friend_name=friend_name, database_="neo4j",
    )

#### Langkah 3: Membaca data dan menjalankan keseluruhan

Fungsi `print_friends(driver, name)` membaca teman dari satu `Person`:
- `MATCH (a:Person)-[:KNOWS]->(friend) WHERE a.name = $name` mencari semua simpul yang dituju relasi `KNOWS` dari `a`.
- `RETURN friend.name ORDER BY friend.name` mengembalikan nama teman, diurutkan secara alfabetis.
- `routing_=RoutingControl.READ` menandai bahwa query ini hanya **membaca**, sehingga pada klaster dapat diarahkan ke server yang melayani pembacaan.

Blok `with GraphDatabase.driver(URI, auth=AUTH) as driver:` membuka koneksi dan menutupnya otomatis setelah selesai. Di dalamnya dibuat tiga pertemanan Arthur (Guinevere, Lancelot, Merlin), kemudian `print_friends` mencetaknya.

In [3]:
def print_friends(driver, name):
    records, _, _ = driver.execute_query(
        "MATCH (a:Person)-[:KNOWS]->(friend) WHERE a.name = $name "
        "RETURN friend.name ORDER BY friend.name",
        name=name, database_="neo4j", routing_=RoutingControl.READ,
    )
    for record in records:
        print(record["friend.name"])

with GraphDatabase.driver(URI, auth=AUTH, ) as driver:
    add_friend(driver, "Arthur", "Guinevere")
    add_friend(driver, "Arthur", "Lancelot")
    add_friend(driver, "Arthur", "Merlin")
    print_friends(driver, "Arthur")

Guinevere
Lancelot
Merlin


**Hasil.** Output menampilkan tiga nama yang terurut menaik (alfabetis), sesuai tiga relasi `KNOWS` yang dibuat dari Arthur.

### Pertanyaan Pendahuluan (Checkpoint)

#### 1. Apa nama database yang digunakan pada tutorial ini?

**Jawaban:** **`neo4j`**.

- Database yang dipakai adalah DB bawaan bernama `neo4j`, terlihat dari `database_="neo4j"` pada `execute_query`.

![Bukti jawaban 1: database neo4j](checkpoint1.png)

---

#### 2. Sebutkan akses lain (seperti sintaks DDL/DML Cypher) yang dapat digunakan untuk mengakses database lain dalam 1 instance

**Jawaban:** satu instance Neo4j dapat memiliki beberapa database. Cara mengakses atau berpindah database lain:

| Konteks | Perintah | Fungsi |
|---|---|---|
| Cypher | `SHOW DATABASES` | melihat daftar database pada instance |
| Neo4j Browser / cypher-shell | `:use system` | masuk ke database administrasi (`system`) |
| Python | `execute_query(..., database_="system")` | memilih database untuk satu query |
| Python | `driver.session(database="system")` | memilih database untuk satu session |

![Bukti jawaban 2: akses database di Neo4j](checkpoint2.png)

---

#### 3. Apa jenis graf pertemanan yang telah dibuat di sini?

**Jawaban:** graf **berarah (directed)** dan **tak berbobot (unweighted)**.

- **Berarah:** relasi `(a)-[:KNOWS]->(friend)` memiliki arah. `Arthur KNOWS Guinevere`.
- **Tak berbobot:** relasi `KNOWS` tidak memiliki bobot.

![Bukti jawaban 3: graf berarah dengan Arthur di pusat](checkpoint3.png)

#### Bukti jawaban nomor 2

Sel berikut menjalankan `SHOW DATABASES` pada database `system` untuk menampilkan daftar database di instance beserta statusnya. Ini menunjukkan bahwa database lain diakses dengan memilih database tujuan lewat parameter `database_`.

In [ ]:
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    records, _, _ = driver.execute_query(
        "SHOW DATABASES", database_="system", routing_=RoutingControl.READ,
    )
    for r in records:
        print(r["name"], "|", r["currentStatus"])

#### Bukti jawaban nomor 3

Sel berikut menampilkan setiap relasi `KNOWS` dalam bentuk `dari -> ke`, beserta properti relasinya. Dua hal terlihat jelas:
- arah relasi selalu dari Arthur ke temannya (graf **berarah**);
- properti relasi kosong (`{}`), sehingga tidak ada bobot (graf **tak berbobot**).

In [ ]:
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    records, _, _ = driver.execute_query(
        "MATCH (a:Person)-[r:KNOWS]->(b:Person) "
        "RETURN a.name AS dari, type(r) AS relasi, b.name AS ke, properties(r) AS properti "
        "ORDER BY ke",
        database_="neo4j", routing_=RoutingControl.READ,
    )
    for r in records:
        print(r["dari"], f"-[{r['relasi']}]->", r["ke"], "| properti:", r["properti"])